# Day 5 - ETL, Tests and Demo

**Goal:** Read raw data, clean it with pandas, and load it into PostgreSQL running in Docker.

This notebook is generic and works with the apprentice's own scraper output.

In [2]:
import os
import json
import hashlib
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

raw_json = Path('raw_data.json')
raw_csv = Path('raw_data.csv')

if not raw_json.exists() and not raw_csv.exists():
    sample = [
        {'scraped_at': '2026-06-30T12:00:00', 'item': 'Example A', 'value_text': '10.5', 'value': None, 'quantity': None, 'source_url': 'https://example.com/a'},
        {'scraped_at': '2026-06-30T12:05:00', 'item': 'Example B', 'value_text': '7.0', 'value': None, 'quantity': 2, 'source_url': 'https://example.com/b'},
    ]
    raw_json.write_text(json.dumps(sample, indent=2), encoding='utf-8')
    print('Wrote sample raw_data.json')

print('Ready')


Wrote sample raw_data.json
Ready


In [3]:
def load_raw(path_json='raw_data.json', path_csv='raw_data.csv'):
    p_json = Path(path_json)
    p_csv = Path(path_csv)
    if p_json.exists():
        with p_json.open('r', encoding='utf-8') as f:
            raw = json.load(f)
        return pd.DataFrame(raw)
    if p_csv.exists():
        return pd.read_csv(p_csv, parse_dates=['scraped_at'], encoding='utf-8')
    return pd.DataFrame()

def parse_number(value):
    if pd.isna(value):
        return None
    try:
        s = str(value).replace(',', '.')
        s = ''.join(ch for ch in s if ch.isdigit() or ch == '.')
        return float(s) if s else None
    except Exception:
        return None

def make_unique_key_row(row):
    parts = [str(row.get('source_url', '')), str(row.get('item', '')).strip().lower(), str(row.get('scraped_at', ''))]
    return hashlib.sha256('|'.join(parts).encode('utf-8')).hexdigest()

def clean_df(df):
    df = df.copy()
    df['scraped_at'] = pd.to_datetime(df.get('scraped_at'), errors='coerce')
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
    else:
        df['value'] = None
    if df['value'].isna().any() and 'value_text' in df.columns:
        df['value'] = df['value'].fillna(df['value_text'].apply(parse_number))
    if 'quantity' in df.columns:
        df['quantity'] = df['quantity'].fillna(1).astype(int)
    else:
        df['quantity'] = 1
    df['total'] = df['value'] * df['quantity']
    df = df[df['item'].notna() & df['value'].notna()]
    df['unique_key'] = df.apply(make_unique_key_row, axis=1)
    df = df.drop_duplicates(subset=['unique_key'])
    return df


In [4]:
df_raw = load_raw()
df_clean = clean_df(df_raw)
df_clean.head()


,scraped_at,item,value_text,value,quantity,source_url,total,unique_key
0,2026-06-30 12:00:00,Example A,10.5,10.5,1,https://example.com/a,10.5,a0e4a21a9403668bdb559dd6d2143d642e594e021bba87...
1,2026-06-30 12:05:00,Example B,7.0,7.0,2,https://example.com/b,14.0,5bb277c33ce8952868244c98743ff69029775fda2ca4c8...


In [ ]:
# PostgreSQL connection settings for the Docker container
db_user = os.environ.get('DB_USER', 'apprentice')
db_pass = os.environ.get('DB_PASS', 'apprentice_pw')
db_host = os.environ.get('DB_HOST', 'localhost')
db_port = os.environ.get('DB_PORT', '5432')
db_name = os.environ.get('DB_NAME', 'apx_data')

conn_str = f'postgresql+psycopg://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}'
engine = create_engine(conn_str)
with engine.connect() as conn:
    print(conn.execute(text('select 1')).fetchone())


In [ ]:
df_clean.to_sql('records', engine, if_exists='replace', index=False)
print('Rows loaded:', len(df_clean))
check = pd.read_sql('select count(*) as row_count from records', engine)
check


## Suggested tests
Try this in a separate test file later:

- one test for parsing numbers
- one test for deduplication

Example idea: create two identical rows and assert the cleaned result has one row.

## Demo checklist
- start the PostgreSQL Docker container
- run the notebook top to bottom
- show the row count query
- explain one cleaning rule and one test idea